# r

In [4]:
# ─────────────────────────────────────────────────────────────
# Loads all classes (FuturesExecutionEnv, DQNAgent, etc.) 
# from your training notebook into memory Futures Execution code.ipynb
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os
import json

from collections import deque

## helps with generating random numbers
import random

from typing import Dict, Tuple, List

## helps with creating Reinforcement learning environments
import gymnasium as gym
from gymnasium import spaces

# file chooser
import tkinter as tk
from tkinter import filedialog
from pathlib import Path

def choose_training_file() -> Path | None:
    root = tk.Tk()
    root.withdraw()

    file_path = filedialog.askopenfilename(
        title="Select training data file",
        filetypes=[
            ("CSV files", "*.csv"),
            ("Text files", "*.txt")
        ]
    )

    return Path(file_path) if file_path else None


def load_training_data(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path)

    elif suffix == ".txt":
        return pd.read_csv(file_path, sep=None, engine="python")

    else:
        raise ValueError("Unsupported file format")


class FuturesExecutionEnv(gym.Env):
    """Custom Environment for Futures Execution"""
    
    ## We are setting up the room i.e. the constructor
    ## initialising order parameters
    ## if parameter values are not given, then the below values will be used
    def __init__(self, data: pd.DataFrame, order_size: int = 1000, time_horizon: int = 60, adv: float = 1000000, side: str = "BUY" ): ## int here is a type hint
        
        ## inheriting core functionality/structure from gym.Env
        ## need the below so we can properly initialize the parent class gym.Env within our FuturesExecutionEnv class
        super().__init__()
        
        assert len(data) >= time_horizon, (
        "Not enough data "
        "Each row must represent one minute, so "
        "len(data) must be >= time_horizon."
        )

        
        self.data = data  # OHLCV (Open High Low Close Volume) + order book data
        self.order_size = order_size
        self.time_horizon = time_horizon  # in minutes
        self.current_step = 0 ## setting up the time-counter to the initial historical point
        self.adv = adv
        self.side = side
        
        # Action space: we require 3 distinctive actions for now: 0=Passive, 1=Moderate, 2=Aggressive
        self.action_space = spaces.Discrete(3)
        
        # State space: we have 8 normalized features and we want to make sure these are normalized
        self.observation_space = spaces.Box(
            low = 0, high = 1, 
            shape=(8,),  # 8 state features
            dtype=np.float32
        )
        
        self.reset() ## resetting everytime
    
    
    
    ## this part is the AI agent's state or dahsboard
    ## it is compiling 8 pieces of normalized information, i.e  between 0 and 1
    def _get_state(self) -> np.ndarray: ## this function should return a numpy array (vector)
        
        """Get current state representation"""
        if self.current_step >= len(self.data) - 1: ## if the index of the current_step reaches the end, then it reverts back to the second last item
            self.current_step = len(self.data) - 2
            
        current_data = self.data.iloc[self.current_step]

        "do we need the below?"
        # next_data = self.data.iloc[self.current_step + 1]
        
        # Normalized state features
        state = np.array([
            self.remaining_quantity / self.order_size,  # % remaining
            self.current_step / self.time_horizon,         # % time elapsed
            np.tanh(current_data['spread']/0.05),        # normalized spread
            np.clip(current_data['volatility'] / 0.02, 0, 1),  # normalized volatility
            (current_data['imbalance'] + 1) / 2,                  # order book imbalance (-1 to 1)
            current_data['volume_ratio'],               # volume ratio
            self._get_urgency(),                        # execution urgency, we will define these functions in a bit
            self._get_performance()                     # current performance , we will define these funcitons in a bit 
        ], dtype=np.float32)
        
        return np.clip(state, 0, 1)                     # this limits the values in the state array between 0 and 1
    
    
    ## early on, we want to focus more on the quantity, 
    ## later on time becomes more important as we run out of trading time
    def _get_urgency(self) -> float:
        
        """Calculate execution urgency based on remaining time/quantity"""
        
        time_urgency = (self.current_step - self.start_step) / self.time_horizon
        quantity_urgency = 1 - (self.remaining_quantity / self.order_size) ## close to 1 during the end
        
        if time_urgency >= 1:
            return 1.0 ## Deadline reached and maximum urgency
        
        ## gets it close to 1 in the beginning and reduces to 0 as time becomes more important later on
        urgency = (
            (1 - time_urgency) * quantity_urgency +
            time_urgency * 1)
        
        return float(np.clip(urgency, 0 , 1))
    


    def _get_performance(self) -> float:
        """Calculate current execution performance"""
    
        if self.quantity_executed == 0:
            return 0.5  # neutral
        
        current_vwap = self.total_value / self.quantity_executed
        
        # Adjust performance calculation based on side
        if self.side == 'BUY':
            arrival_performance = (self.arrival_price - current_vwap) / self.arrival_price
        else:  # SELL
            arrival_performance = (current_vwap - self.arrival_price) / self.arrival_price
        
        return np.clip(arrival_performance * 10 + 0.5, 0, 1)
    

    def _calculate_market_impact(self, action: int, quantity: int, spread: float, depth_factor: float = 0.5) -> float:
        """Realistic Market Impact model"""
   
    
    # --- 1. Passive Order → No spread, no impact ---
        if action == 0:
            return 0.0

    # Normalized volume relative to ADV (ADV stored in environment)
        vol_fraction = quantity / self.adv
    # --- 2. Transient Market Impact Component (exponential response) ---
    # η controls sensitivity:  Aggressive > Moderate > Passive
        eta = [0.0, 0.6, 1.2][action]   # you can tune these three values later
        transient_impact = eta * (vol_fraction ** 0.6)

        # --- 3. Spread Cost (only when crossing the book) ---
        # Half-spread cost for a market buy/sell
        spread_cost = spread * 0.5

        # Moderate orders cross less of the spread
        if action == 1:
            spread_cost *= 0.4      # moderate pays only ~40% of spread

        # --- 4. Depth Cost (slippage due to sweeping orderbook) ---get
        # Only for aggressive orders
        depth_cost = 0.0
        if action == 2:
            depth_cost = depth_factor * (vol_fraction ** 1.2)

        # --- Final Impact ---
        total_impact = spread_cost + transient_impact + depth_cost
        return total_impact


    def _get_execution_price(self, action: int, mid_price: float, spread: float) -> float:
        """Get execution price based on action type"""
        ## raw execution price without the market impact

        direction = 1 if self.side == "BUY" else -1

        if action == 0:      # passive
            return mid_price - direction * spread * 0.3
        elif action == 1:    # moderate
            return mid_price - direction * spread * 0.1
        else:                # aggressive
            return mid_price + direction * spread * 0.5
   
    
    def step(self, action: int):

        current_data = self.data.iloc[self.current_step]
        mid_price = current_data['close']
        spread = current_data['spread']

        base_quantity = max(1, int(self.order_size * 0.1))
        quantity_multiplier = {0: 0.3, 1: 0.6, 2: 1}

        quantity = min(int(base_quantity * quantity_multiplier[action]),self.remaining_quantity)

        if quantity > 0:
            exec_price = self._get_execution_price(action, mid_price, spread)
            impact = self._calculate_market_impact(action, quantity, spread)
            final_price = exec_price + impact

            self.quantity_executed += quantity
            self.remaining_quantity -= quantity
            self.total_value += final_price * quantity

        reward = self._calculate_reward(action, quantity, mid_price, spread)

        self.current_step += 1

        terminated = self.remaining_quantity <= 0
        truncated = (self.current_step - self.start_step) >= self.time_horizon

        info = {
            'step': self.current_step,
            'quantity_executed': self.quantity_executed,
            'remaining_quantity': self.remaining_quantity,
            'avg_price': self.total_value / self.quantity_executed if self.quantity_executed > 0 else 0,
            'action': action,
            'reward': reward
        }
        # Return 5 values as required by gymnasium
        return self._get_state(), reward, terminated, truncated, info
    
    def _calculate_reward(self, action: int, quantity: int, mid_price: float, spread: float) -> float:
        """Calculate reward for the action taken"""
        
        if quantity == 0:
            return -0.5 if self._get_urgency() > 0.7 else -0.05 ## higher penalty for no execution later
    
        # Get execution price (same logic as in step())
        exec_price = self._get_execution_price(action, mid_price, spread)

        # Slippage THIS STEP only
        if self.side == "BUY":
            step_slippage = (exec_price - mid_price) / mid_price*10000 ## in basis points
        else:
            step_slippage = (mid_price - exec_price) / mid_price*10000 ## in basis points

            # Convert to reward (negative slippage = bad)
        price_reward = -step_slippage * 0.2
        
        # Penalty for market impact
        impact_penalty = -self._calculate_market_impact(action, quantity, spread) * 0.5
        
        # progres bonus
        progress = self.quantity_executed / self.order_size
        quality = self._get_performance()
        progress_bonus = progress * (quality - 0.5) * 0.5
        
        
        # Urgency bonus/penalty
        urgency = self._get_urgency()
        if progress < 0.5 and urgency > 0.8:
            time_penalty = -0.3 # strong penalty
        elif urgency < 0.8 and progress > 0.6:
            time_penalty = -0.1
        else:
            time_penalty = 0.0  # Bonus for completing early
        
        # Risk penalty for large remaining quantity late in execution
        risk_penalty = - (self.remaining_quantity / self.order_size) * max(0, urgency-0.5)*0.3
        
        # completion penalty
        completion_bonus = 2.0 if self.remaining_quantity == 0 else 0.0
        
        total_reward = price_reward + impact_penalty + time_penalty + risk_penalty + completion_bonus + progress_bonus
        return float(np.clip(total_reward, -5, 5))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.start_step = np.random.randint(0, len(self.data) - self.time_horizon)
        self.current_step = self.start_step   # ← add this line
        self.remaining_quantity = self.order_size
        self.quantity_executed = 0
        self.total_value = 0.0
        self.arrival_price = self.data.iloc[self.current_step]['close']
        return self._get_state(), {}
    
## Start of the NN logic

class DQNAgent:

    ## Q(s,a) => Expected future reward if I take action "a" in state "s"
    def __init__(self, state_size: int, action_size: int):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=100000) ## can remember the latest 2000 recent  steps
        self.gamma = 0.95  # discount rate, anything close to 1 means that the function cares more about future rewards
        
        # Start: ε = 1 → 100% random actions
        # Agent knows nothing, so explores everywhere.
        # Decay: ε *= 0.995 after each step / episode
        # Slowly prefers exploitation.
        # End: ε = 0.01 → 1% random actions
        # Almost always picks best action (according to Q-values), but still occasionally explores to avoid local optima.

        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

        self.target_update_freq  = 200
        self.step_count = 0
        self.learning_rate = 0.001 ##small number means slow, stable learning and changes mind slowly
        self.model = self._build_model() ## this is main brain function we will define next, the Q-function
        self.target_model = self._build_model() ## this is the RHS of the Bellman eqn we want to replace with as the actual function keeps on changing weights.
        self.update_target_network() ## we will update the target model (RHS of Bellman eqn) after some steps as given in the function

        # NEW: Training counters for batch processing
        self.learn_step_counter = 0
        self.train_frequency = 4  # Learn every 4 steps
        
    
    def _build_model(self):
        """Build neural network model"""
        from tensorflow.keras.models import Sequential ## Used for empty model container, we can store model here
        from tensorflow.keras.layers import Dense, BatchNormalization, Input
        from tensorflow.keras.optimizers import Adam

        ## this a Q-network that approximates the Q-function
        ## Predict expected future reward for each action
        ## the idea is that, we have a target Q-value from Bellman eqn and we use DQN function approximator (adj weights dynamically) to spit out Q-value closest to the target
        ## Updates weights to minimize difference between predicted(using NN) and actual Q-values (Bellmann Eqn)
        
        model = Sequential([
            Input(shape=(self.state_size,)),  # Explicit Input layer
            Dense(64, activation='relu'),
            BatchNormalization(),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(self.action_size, activation="linear")
        ])
    
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model
    
    def update_target_network(self):
        ## Update target network weights Q(s,a) after each transition, we want the NN to stay updated and learn
        self.target_model.set_weights(self.model.get_weights()) ## these functions are defined in TensowFlow memories
    
    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay memory"""
        # Ensure state is 1D array
        if isinstance(state, np.ndarray) and state.shape == (1, self.state_size):
            state = state.flatten()
        if isinstance(next_state, np.ndarray) and next_state.shape == (1, self.state_size):
            next_state = next_state.flatten()
    
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state: np.ndarray) -> int:
        """Choose action using epsilon-greedy policy"""
        if np.random.random() <= self.epsilon:
            return random.randrange(self.action_size)

        # Ensure state is 2D for prediction but store as 1D
        state_2d = state.reshape(1, -1) if len(state.shape) == 1 else state
        act_values = self.model.predict(state_2d, verbose=0)
        return np.argmax(act_values[0])
    
    def replay(self, batch_size: int = 32):

        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        
        states = np.array([s[0] for s in minibatch])
        next_states = np.array([s[3] for s in minibatch])
        actions = np.array([s[1] for s in minibatch])
        rewards = np.array([s[2] for s in minibatch])
        dones = np.array([s[4] for s in minibatch])

       # ENRICHMENT #1: VECTORIZED PREDICTIONS
        # Single batch prediction instead of multiple
        current_q = self.model.predict(states, verbose=0)
        next_q = self.target_model.predict(next_states, verbose=0)
        
        # ENRICHMENT #1: VECTORIZED TARGET COMPUTATION
        # This replaces the slow Python loop with fast NumPy operations
        max_next_q = np.max(next_q, axis=1)
        
        # Vectorized Bellman update
        # targets = rewards + gamma * max_next_q (for non-terminal states)
        targets = rewards + self.gamma * max_next_q * (1 - dones)
        
        # Update Q-values for the actions taken
        for i, action in enumerate(actions):
            current_q[i][action] = targets[i]
        
        # ENRICHMENT #3: BATCH LEARNING
        # Single training step on the entire batch
        self.model.fit(states, current_q, batch_size=batch_size, 
                      epochs=1, verbose=0)
        
        self.step_count += 1
        
        # Update target network periodically
        if self.step_count % self.target_update_freq == 0:
            self.update_target_network()

    def load(self, name):
        self.model.load_weights(name)
        meta_path = name.replace('.h5', '_meta.json')
        if os.path.exists(meta_path):
            with open(meta_path, 'r') as f:
                meta = json.load(f)
            self.epsilon = meta.get('epsilon', self.epsilon_min)
            print(f"Resumed with epsilon: {self.epsilon:.3f}")
        else:
            print(f"No metadata found for {name}, using current epsilon: {self.epsilon:.3f}")


    def save(self, name):
        """Save model weights and metadata"""
        # Save weights
        self.model.save_weights(name)
        
        # Save metadata
        meta = {'epsilon': self.epsilon}
        meta_path = name.replace('.h5', '_meta.json')
        with open(meta_path, 'w') as f:
            json.dump(meta, f)
        print(f"Model saved to {name} with epsilon={self.epsilon:.3f}")

 ## Actual futures trading algorithm           
class FuturesExecutionAlgo:
    """Main Futures Execution Algorithm"""
    
    def __init__(self, symbol: str = "ES"):
        self.symbol = symbol
        self.agent = None
        self.env = None
        self.is_trained = False
        self.state_size = None
        self.action_size = None
    
    def prepare_data(self, raw_data: pd.DataFrame) -> pd.DataFrame:
        """Prepare and feature engineer market data"""
        data = raw_data.copy()
        
        ## Data column validation
        required_cols = ['close', 'bid', 'ask', 'bid_volume', 'ask_volume', 'volume']
        missing = [col for col in required_cols if col not in data.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        
        # Calculate features
        data['returns'] = data['close'].pct_change()
        data['volatility'] = data['returns'].rolling(20).std().fillna(0.01)
        data['spread'] = (data['ask'] - data['bid']).fillna(0.25)
       
        ## checking the bid and ask demand, if anything close to 1 then it is bid pressure
        data['imbalance'] = ((data['bid_volume'] - data['ask_volume']) / 
                           (data['bid_volume'] + data['ask_volume'] + 1e-8)).fillna(0)
        
        ## current volume divided recent average volume
        data['volume_ratio'] = (data['volume'] / data['volume'].rolling(50).mean().replace(0, np.nan)).fillna(1)
            
        # Normalize features
        data['volatility'] = data['volatility'].clip(0, 0.02)
        data['spread'] = data['spread'].clip(0.1, 2.0)
        
        data = data.dropna()
        
        # Ensure we have enough data
        if len(data) < 60:  # Minimum time horizon
            raise ValueError(f"Not enough data after preprocessing. Got {len(data)} rows, need at least 60")
        
        return data
    
    def train(self, data: pd.DataFrame, episodes: int = 500):
        """Train the RL agent"""
        print("Starting training...")
        
        # Prepare environment
        processed_data = self.prepare_data(data)
        self.env = FuturesExecutionEnv(processed_data)
        
        # Initialize agent
        state_size = self.env.observation_space.shape[0]
        action_size = self.env.action_space.n
        self.agent = DQNAgent(state_size, action_size)

        # Load existing model if available
        weights_path = "futures_execution_model.weights.h5"
        meta_path = "futures_execution_model_meta.json"

        if os.path.exists(weights_path):
            print("Loading existing trained model...")
            try:
                self.agent.load(weights_path)
            except Exception as e:
                print(f"Error loading model: {e}")
                print("Training from scratch.")
        else:
            print("No existing model found. Training from scratch.")
        

        # ENRICHMENT #3: BATCH PROCESSING PARAMETERS
        BATCH_SIZE = 32
        LEARNING_STARTS = 200  # Start learning after collecting enough experiences
        TRAIN_FREQ = 4  # Train every 4 steps
        
        # Training metrics
        scores = []
        step_count = 0
        
        for episode in range(episodes):
            state, _ = self.env.reset()
            if len(state.shape) == 1:
                state = state.reshape(1, -1)
            
            total_reward = 0
            done = False
            truncated = False
            episode_steps = 0
            
            while not (done or truncated):
                # Select action
                action = self.agent.act(state)
                
                # Take action
                next_state, reward, terminated, truncated, info = self.env.step(action)
                done = terminated
                
                if next_state is not None:
                    if len(next_state.shape) == 1:
                        next_state = next_state.reshape(1, -1)
                    
                    # Store experience
                    self.agent.remember(state, action, reward, next_state, done)
                
                state = next_state
                total_reward += reward
                episode_steps += 1
                step_count += 1
                
                # ENRICHMENT #3: BATCHED LEARNING
                # Only start learning after collecting enough experiences
                if (step_count > LEARNING_STARTS and 
                    step_count % TRAIN_FREQ == 0 and 
                    len(self.agent.memory) > BATCH_SIZE):
                    
                    # Use the agent's replay method with vectorized operations
                    self.agent.replay(BATCH_SIZE)
            
            scores.append(total_reward)
            
            if self.agent.epsilon > self.agent.epsilon_min:
                self.agent.epsilon *= self.agent.epsilon_decay  # now decays once per episode, not per step
            
            # Progress tracking
            if episode % 50 == 0:
                avg_score = np.mean(scores[-50:]) if scores else 0
                print(f"Episode {episode}, Score: {total_reward:.2f}, "
                      f"Avg (50): {avg_score:.2f}, Epsilon: {self.agent.epsilon:.3f}, "
                      f"Memory: {len(self.agent.memory)}")
        
        self.is_trained = True
        self.agent.save("futures_execution_model.weights.h5")
        print("Training completed!")
        return scores
    
    def execute_order(self, live_data: pd.DataFrame, order_size: int) -> Dict:
        """Execute a live order using trained agent"""
            
        ## safety check, not allowed to trade unless the AI is trained 
        if not self.is_trained:
            raise ValueError("Agent must be trained before execution")
        
        ## data gets processed, this method was created earlier where different features were created for a dataset
        processed_data = self.prepare_data(live_data)
            
        ## Create a brand-new execution environment using this market data and this order size, and store it in self.env
        ## class is instantiated
        self.env = FuturesExecutionEnv(processed_data, order_size=order_size)
        
        state, _ = self.env.reset() ## resetting
        done = False ## nothing executed yet
        truncated = False
        execution_log = [] ## notebook to log actions
        
        while not (done or truncated): ## keep trading until "done"
            action = self.agent.act(state) ## AI looks at features and decides what action to take
            
            ## trade gets executed, moves to next step, collates info like reward, price, how much traded, how much left
            next_state, reward, terminated, truncated, info = self.env.step(action)
            done = terminated or truncated # combine for loop condition
            
            ## details get logged
            execution_log.append({
                'step': info['step'],
                'action': action,
                'quantity_executed': info['quantity_executed'],
                'remaining_quantity': info['remaining_quantity'],
                'average_price': info['avg_price'],
                'reward': reward
            })
            
            state = next_state ## update view of the market

        # Correct implementation shortfall calculation based on side (BUY/SELL)
        if self.env.side == "BUY":
            implementation_shortfall = (self.env.arrival_price - info['avg_price']) * order_size
        else:  # SELL
            implementation_shortfall = (info['avg_price'] - self.env.arrival_price) * order_size
        
        # Execution summary
        summary = {
            'total_quantity': order_size,
            'executed_quantity': info['quantity_executed'],
            'average_execution_price': info['avg_price'],
            'arrival_price': self.env.arrival_price,
            'implementation_shortfall': (self.env.arrival_price - info['avg_price']) * order_size,
            'completion_rate': info['quantity_executed'] / order_size,
            'execution_log': execution_log
        }
        
        print(f"Average Price: {summary['average_execution_price']:.2f}")
        print(f"Arrival Price: {summary['arrival_price']:.2f}")
        print(f"Implementation Shortfall: ${summary['implementation_shortfall']:.2f}")

        return summary

##################################################################################################
# # Clean import — only loads the 3 classes from above, nothing executes
# from futures_classes import FuturesExecutionEnv, DQNAgent, FuturesExecutionAlgo

################################################################################################

import pandas as pd
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

def run_execution_test(
    order_size:  int  = 100,
    side:        str  = "BUY",
    model_path:  str  = "futures_execution_model.weights.h5",
    show_log:    bool = True,
):
    # ── 1. File selection ────────────────────────────────────────────────────
    root = tk.Tk()
    root.withdraw()
    file_path = Path(filedialog.askopenfilename(title="Select market data CSV"))
    if not file_path.exists():
        raise FileNotFoundError("No file selected.")

    raw_data = pd.read_csv(file_path)
    print(f"Loaded : {file_path.name}  ({len(raw_data)} rows)")

    # ── 2. Initialise algo and load model ────────────────────────────────────
    algo = FuturesExecutionAlgo(symbol="ES")

    processed_data = algo.prepare_data(raw_data)
    _tmp_env = FuturesExecutionEnv(processed_data, order_size=order_size, side=side)

    algo.agent = DQNAgent(
        state_size=_tmp_env.observation_space.shape[0],
        action_size=_tmp_env.action_space.n,
    )
    algo.agent.load(model_path)
    algo.agent.epsilon = 0.0
    algo.is_trained    = True

    print(f"Model  : {model_path}")
    print(f"Side   : {side}  |  Order size: {order_size} contracts\n")

    # ── 3. Execution with side injection ─────────────────────────────────────
    def _execute_with_side(live_data, order_size):
        algo.env = FuturesExecutionEnv(
            algo.prepare_data(live_data),
            order_size=order_size,
            side=side,
        )
        state, _ = algo.env.reset()
        done       = False
        truncated  = False
        execution_log = []

        # Track previous step values to compute per-step execution price
        prev_qty   = 0
        prev_value = 0.0

        while not (done or truncated):
            action = algo.agent.act(state)
            state, reward, terminated, truncated, info = algo.env.step(action)
            done = terminated or truncated

            # ── Bid / Ask at this step ───────────────────────────────────────
            # current_step has already advanced by 1 inside step(), so we
            # look back one index to get the data for the step just executed
            step_idx  = algo.env.current_step - 1
            step_data = algo.env.data.iloc[step_idx]
            bid       = step_data['bid']
            ask       = step_data['ask']

            # ── Per-step execution price ─────────────────────────────────────
            # avg_price in info is cumulative VWAP, not this step's price.
            # We back it out from the change in total value and quantity.
            current_qty   = info['quantity_executed']
            current_value = info['avg_price'] * current_qty   # total value so far

            qty_this_step = current_qty - prev_qty
            if qty_this_step > 0:
                step_exec_price = (current_value - prev_value) / qty_this_step
            else:
                step_exec_price = step_data['close']          # no trade, show mid

            prev_qty   = current_qty
            prev_value = current_value

            execution_log.append({
                'step':               info['step'],
                'action':             action,
                'qty_this_step':      qty_this_step,           # contracts traded THIS step
                'quantity_executed':  info['quantity_executed'],
                'remaining_quantity': info['remaining_quantity'],
                'bid':                bid,
                'ask':                ask,
                'step_exec_price':    step_exec_price,         # price THIS step traded at
                'cumulative_vwap':    info['avg_price'],       # running average
                'reward':             reward,
            })

        # Side-correct implementation shortfall
        exec_qty  = info['quantity_executed']
        avg_price = info['avg_price']
        arrival   = algo.env.arrival_price

        if side == "BUY":
            shortfall = (arrival - avg_price) * exec_qty
        else:
            shortfall = (avg_price - arrival) * exec_qty

        return {
            'total_quantity':           order_size,
            'executed_quantity':        exec_qty,
            'average_execution_price':  avg_price,
            'arrival_price':            arrival,
            'implementation_shortfall': shortfall,
            'completion_rate':          exec_qty / order_size,
            'execution_log':            execution_log,
        }

    # ── 4. Run execution ─────────────────────────────────────────────────────
    summary = _execute_with_side(raw_data, order_size)
    log     = summary['execution_log']

    ACTION_LABELS = {0: "Passive", 1: "Moderate", 2: "Aggressive"}

    # ── 5. Step-by-step table ────────────────────────────────────────────────
    if show_log:
        print(
            f"{'Step':>5}  {'Action':>10}  {'Qty':>4}  {'Exec Total':>10}  "
            f"{'Remaining':>9}  {'Bid':>10}  {'Ask':>10}  "
            f"{'Step Px':>10}  {'VWAP':>10}  {'Reward':>8}"
        )
        print("─" * 105)

        for e in log:
            print(
                f"{e['step']:>5}  "
                f"{ACTION_LABELS[e['action']]:>10}  "
                f"{e['qty_this_step']:>4}  "
                f"{e['quantity_executed']:>10}  "
                f"{e['remaining_quantity']:>9}  "
                f"{e['bid']:>10.4f}  "
                f"{e['ask']:>10.4f}  "
                f"{e['step_exec_price']:>10.4f}  "   # price THIS step traded at
                f"{e['cumulative_vwap']:>10.4f}  "   # running average
                f"{e['reward']:>8.3f}"
            )

    # ── 6. Action breakdown ──────────────────────────────────────────────────
    actions     = [e['action'] for e in log]
    total_steps = len(actions)
    breakdown   = {label: actions.count(i) for i, label in ACTION_LABELS.items()}

    # ── 7. Final summary ─────────────────────────────────────────────────────
    exec_qty      = summary['executed_quantity']
    shortfall_per = summary['implementation_shortfall'] / exec_qty if exec_qty else 0
    price_vs_arr  = summary['average_execution_price'] - summary['arrival_price']
    if side == "SELL":
        price_vs_arr = -price_vs_arr

    avg_spread = np.mean([(e['ask'] - e['bid']) for e in log])

    print(f"\n{'═' * 48}")
    print(f"  EXECUTION SUMMARY  ({side})")
    print(f"{'═' * 48}")
    print(f"  Contracts Ordered    : {order_size}")
    print(f"  Contracts Executed   : {exec_qty}")
    print(f"  Completion Rate      : {summary['completion_rate']*100:.1f}%")
    print(f"  Arrival Price        : {summary['arrival_price']:.4f}")
    print(f"  Avg Exec Price       : {summary['average_execution_price']:.4f}")
    print(f"  Price vs Arrival     : {price_vs_arr:+.4f} pts")
    print(f"  Impl. Shortfall      : ${summary['implementation_shortfall']:.2f}")
    print(f"  Shortfall / Contract : ${shortfall_per:.4f}")
    print(f"  Avg Bid-Ask Spread   : {avg_spread:.4f}")
    print(f"  Steps Taken          : {total_steps}")
    print(f"{'─' * 48}")
    print(f"  Action Breakdown:")
    for label, count in breakdown.items():
        bar = "█" * int(count / total_steps * 20) if total_steps else ""
        print(f"    {label:<11}: {count:>4} steps "
              f"({count/total_steps*100:4.1f}%)  {bar}")
    print(f"{'═' * 48}\n")

    return summary


# ── Entry point ───────────────────────────────────────────────
summary = run_execution_test(
    order_size=100,
    side="BUY",
    model_path="futures_execution_model.weights.h5",
    show_log=True,
)


Loaded : synthetic_futures_seed_112.csv  (3000 rows)
Resumed with epsilon: 0.010
Model  : futures_execution_model.weights.h5
Side   : BUY  |  Order size: 100 contracts



C:\Users\sadee\Downloads\my_downloads\envs\tf310\lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


 Step      Action   Qty  Exec Total  Remaining         Bid         Ask     Step Px        VWAP    Reward
─────────────────────────────────────────────────────────────────────────────────────────────────────────
 1227  Aggressive    10          10         90   4600.6573   4600.9073   4601.0335   4601.0335    -0.117
 1228  Aggressive    10          20         80   4609.8365   4610.0865   4610.2127   4605.6231    -0.118
 1229  Aggressive    10          30         70   4612.3244   4613.0744   4613.4506   4608.2322    -0.353
 1230  Aggressive    10          40         60   4615.3745   4616.1245   4616.5007   4610.2994    -0.355
 1231  Aggressive    10          50         50   4617.2511   4618.0011   4618.3773   4611.9149    -0.362
 1232  Aggressive    10          60         40   4620.6964   4620.9464   4621.0726   4613.4412    -0.141
 1233  Aggressive    10          70         30   4620.2206   4620.4706   4620.5968   4614.4634    -0.248
 1234  Aggressive    10          80         20   4623.